In [6]:
import csv
from pathlib import Path
import cv2
import mediapipe as mp
import pandas as pd

# Set up MediaPipe
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

columns = ["frame", "hand", "landmark", "x", "y", "z"]

In [11]:
def extract(video_path, model_path="models/hand_landmarker.task", show_video=False):
    options = HandLandmarkerOptions(
        base_options=BaseOptions(
            model_asset_path="models/hand_landmarker.task"
        ),
        running_mode=RunningMode.VIDEO,
        num_hands=2
    )

    # Open your ASL video
    video = cv2.VideoCapture("videos/ASL_How.mp4")
    fps = video.get(cv2.CAP_PROP_FPS)

    if not video.isOpened():
        raise RuntimeError("Could not open video")

    fps = video.get(cv2.CAP_PROP_FPS) or 30  # fall back if fps is missing
    rows = []
    frame_number = 0


    with HandLandmarker.create_from_options(options) as detector:
        while True:
            success, frame = video.read()

            if not success:
                break

            # Convert frame to RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp = int(frame_number * 1000 / fps)

            # Detect hands
            result = detector.detect_for_video(mp_image, timestamp)

            height, width = frame.shape[:2]

            for hand_index, hand in enumerate(result.hand_landmarks):
                for landmark_id, point in enumerate(hand):
                    rows.append((frame_number, hand_index, landmark_id, point.x, point.y, point.z))

                    if show_video:
                        cv2.circle(
                            frame,
                            (int(point.x * width), int(point.y * height)),
                            4,
                            (0, 255, 0),
                            -1,
                        )
 
            if show_video:
                cv2.imshow("Hand landmarks", frame)
                # Press Q to exit early
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
 
            frame_number += 1
 
    video.release()
    cv2.destroyAllWindows()
 
    return pd.DataFrame(rows, columns=columns)

In [10]:
df = extract("videos/ASL_How.mp4", show_video=False)
df

,frame,hand,landmark,x,y,z
0,25,0,0,0.433568,0.988808,-2.102732e-07
1,25,0,1,0.456868,0.943857,4.495747e-04
2,25,0,2,0.475274,0.873870,1.648063e-03
3,25,0,3,0.477490,0.810278,-1.179680e-03
4,25,0,4,0.473560,0.767133,-2.512419e-03
...,...,...,...,...,...,...
877,49,1,16,0.522630,0.925401,-5.720333e-02
878,49,1,17,0.488024,1.010010,-4.839373e-02
879,49,1,18,0.543132,0.933996,-6.183194e-02
880,49,1,19,0.547711,0.945525,-6.365845e-02


In [13]:
"""Turn one clip's long-format landmark DataFrame (frame, hand, landmark, x, y, z)
into model input.

- clip_to_sequence(df)  -> (n_frames, F) array          (for LSTM / CNN / hybrid)
- clip_to_row(df)       -> (1 row) DataFrame of features (for SVM / random forest)
"""
import numpy as np
import pandas as pd

N_HANDS, N_LANDMARKS = 2, 21


def df_to_array(df):
    """Long format -> (T, 2, 21, 3) array. NaN wherever a hand wasn't detected."""
    if df.empty:
        raise ValueError("No landmarks in this clip (no hands detected).")
    first, last = df["frame"].min(), df["frame"].max()
    arr = np.full((last - first + 1, N_HANDS, N_LANDMARKS, 3), np.nan)
    arr[df["frame"].values - first, df["hand"].values, df["landmark"].values] = df[["x", "y", "z"]].values
    return arr


def fill_gaps(arr, min_frac=0.3):
    """Interpolate short detection gaps. A hand seen in fewer than min_frac of the
    frames is treated as 'not used' and set to zeros."""
    out = arr.copy()
    t = np.arange(len(out))
    for h in range(N_HANDS):
        present = ~np.isnan(out[:, h, 0, 0])  # all 21 landmarks appear together
        if present.mean() < min_frac:
            out[:, h] = 0.0
            continue
        for l in range(N_LANDMARKS):
            for c in range(3):
                out[:, h, l, c] = np.interp(t, t[present], out[present, h, l, c])
    return out


def normalize(arr):
    """Per frame and hand: hand shape relative to the wrist, scaled by hand size
    (wrist -> middle-finger base). Wrist position (x, y) is kept as separate features
    because where the hand is matters for signs."""
    T = len(arr)
    rel = arr - arr[:, :, 0:1, :]
    size = np.linalg.norm(arr[:, :, 9, :2] - arr[:, :, 0, :2], axis=-1)  # (T, 2)
    size = np.where(size > 1e-6, size, 1.0)
    rel = rel / size[:, :, None, None]
    wrist_pos = arr[:, :, 0, :2]                                          # (T, 2, 2)
    return np.concatenate([rel.reshape(T, -1), wrist_pos.reshape(T, -1)], axis=1)  # (T, 130)


def resample(seq, n_frames=30):
    """Stretch/squeeze a (T, F) sequence to exactly n_frames."""
    old, new = np.linspace(0, 1, len(seq)), np.linspace(0, 1, n_frames)
    return np.stack([np.interp(new, old, seq[:, i]) for i in range(seq.shape[1])], axis=1)


def clip_to_sequence(df, n_frames=30):
    """(n_frames, F) array for the neural models."""
    return resample(normalize(fill_gaps(df_to_array(df))), n_frames)


def clip_to_row(df, n_frames=30, mode="summary", **meta):
    """One-row DataFrame for the baseline models.

    mode="summary": mean, std, min, max of each feature over time (4 * F values)
    mode="flatten": every frame's features side by side (n_frames * F values)
    **meta: extra columns placed first, e.g. clip_id="KEY_P2_S1_04", label="KEY", signer="P2"
    """
    seq = clip_to_sequence(df, n_frames)
    if mode == "summary":
        feats = np.concatenate([seq.mean(0), seq.std(0), seq.min(0), seq.max(0)])
    elif mode == "flatten":
        feats = seq.ravel()
    else:
        raise ValueError("mode must be 'summary' or 'flatten'")
    feat_df = pd.DataFrame([feats], columns=[f"f{i}" for i in range(len(feats))])
    return pd.concat([pd.DataFrame([meta]), feat_df], axis=1) if meta else feat_df

In [14]:
clip_to_row(df)

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f510,f511,f512,f513,f514,f515,f516,f517,f518,f519
0,0.0,0.0,0.0,-0.083155,-0.364296,0.008116,-0.098503,-0.751062,-0.008427,-0.09798,...,0.582522,0.457248,-0.638963,0.312738,0.695353,-0.652948,0.644489,1.019867,0.551492,0.954426


In [16]:
pd.DataFrame(clip_to_sequence(df))

,0,1,2,3,4,5,6,7,8,9,...,120,121,122,123,124,125,126,127,128,129
0,0.0,0.0,0.0,0.126508,-0.244062,0.002442,0.226440,-0.624052,0.008949,0.238473,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.433568,0.988808,0.497030,0.910005
1,0.0,0.0,0.0,0.133348,-0.268302,0.002122,0.239284,-0.643707,0.004846,0.283741,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.454286,0.954236,0.497030,0.910005
2,0.0,0.0,0.0,0.138228,-0.285607,0.001893,0.248449,-0.657721,0.001915,0.316066,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.475004,0.919663,0.497030,0.910005
3,0.0,0.0,0.0,0.106729,-0.306846,-0.004885,0.202407,-0.671618,-0.015161,0.283691,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.496395,0.894805,0.497030,0.910005
4,0.0,0.0,0.0,0.019735,-0.358203,-0.004853,0.084036,-0.712607,-0.027487,0.167342,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.528305,0.888628,0.497030,0.910005
5,0.0,0.0,0.0,-0.102025,-0.438318,0.013952,-0.072342,-0.785176,-0.012526,-0.003982,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.572444,0.902001,0.497030,0.910005
6,0.0,0.0,0.0,-0.148633,-0.445532,0.030754,-0.138879,-0.798001,0.013260,-0.115242,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.594080,0.915288,0.497030,0.910005
7,0.0,0.0,0.0,-0.132307,-0.393414,0.025670,-0.156101,-0.797531,0.013950,-0.142755,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.601002,0.925078,0.497030,0.910005
8,0.0,0.0,0.0,-0.128049,-0.356673,0.008478,-0.167405,-0.772679,-0.007348,-0.172248,...,0.506009,-0.228780,-0.741067,0.312738,-0.010517,-0.717969,0.609766,0.945450,0.497030,0.910005
9,0.0,0.0,0.0,-0.117360,-0.345517,0.004503,-0.158747,-0.751231,-0.013058,-0.177791,...,0.501594,-0.278195,-0.768957,0.295218,-0.047003,-0.756237,0.613985,0.966212,0.506670,0.914298


In [ ]:
data\ASL Data Collection\ASL_CIAN\Book\book1.mp4 